# Day 8 - Merge the LoRA adapter and publish to the Hub

Folds the adapter into the base weights and pushes the result. Reasoning behind every step:
`docs/guides/05-merging-and-publishing-a-model.md`.

**Runs on Kaggle rather than locally** because the merge needs ~7 GB of RAM for the fp16 base
model, and it keeps a 6.4 GB download off your machine entirely - the adapter is already here
in the training notebook's output.

**No GPU needed.** Merging is a handful of matrix additions, not a training run. Use a CPU
session and save your GPU quota.

## Before running

1. **Add Data -> Notebook Output** -> select the training notebook, so its `lora-adapter/`
   folder is mounted under `/kaggle/input`.
2. `HF_TOKEN` must be a Kaggle Secret (Add-ons -> Secrets) - the same one used for training.
   It needs **write** permission to push, and access to the gated Llama-3.2 repo.
3. Internet **On**.
4. Set `REPO_ID` below. The name **must begin with `Llama`** - the Llama 3.2 Community
   License requires it for any distributed derivative model.

In [ ]:
# peft is the only thing missing from the stock image. --no-deps stops pip from resolving
# its torch requirement and downgrading Kaggle's torch.
!pip install -q --no-deps peft

# The model card lives in the repo, not in this notebook, so the card and the code that
# generates it cannot drift apart. Pulled from the public repo rather than re-uploaded.
!wget -q -O /kaggle/working/merge.py https://raw.githubusercontent.com/Himanshu7240/finance-chatbot/main/src/training/merge.py

import sys
sys.path.insert(0, "/kaggle/working")
from merge import model_card
print("model_card() loaded from the repo")

In [ ]:
from pathlib import Path

REPO_ID = "Himanshu7240/Llama-3.2-3B-finance-india"   # must start with "Llama"
BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"
OUT = Path("/kaggle/working/merged")

assert REPO_ID.split("/")[-1].lower().startswith("llama"), (
    "Llama 3.2 Community License requires distributed derivative model names to begin "
    "with 'Llama'")

# Find the adapter wherever Kaggle mounted the notebook output.
configs = [p for p in Path("/kaggle/input").rglob("adapter_config.json")
           if "checkpoint" not in str(p)]
assert configs, "no adapter found - did you Add Data -> Notebook Output?"
ADAPTER_DIR = configs[0].parent
print("adapter:", ADAPTER_DIR)
for f in sorted(ADAPTER_DIR.iterdir()):
    print(f"  {f.name:32} {f.stat().st_size / 1e6:8.1f} MB")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("logged in to Hugging Face")

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# fp16, NOT the 4-bit config training used. Merging into quantized weights re-quantizes the
# adapter's contribution through a 4-bit grid and rounds the fine-tuning signal away - and it
# shows up as slightly worse output, never an error. See Guide 05.
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.float16, low_cpu_mem_usage=True, token=HF_TOKEN)
print(f"base loaded, {model.get_memory_footprint() / 1e9:.2f} GB")

model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
model = model.merge_and_unload()
print("merged")

tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR))

In [ ]:
# Smoke-test the merged weights BEFORE publishing. If the merge silently degraded the model,
# this is where it shows - the answer should be the terse extracted span the fine-tune
# produces, not a chatty base-model sentence.
prompt = ("Question: How much might Tata Steel need to pay as minerals tax dues to Odisha?\n"
          "Context: Tata Steel might need to pay more than Rs 17,000 crore as minerals tax "
          "dues to the state of Odisha, after the Supreme Court allowed states to levy taxes "
          "on mineral rights with retrospective effect from April 2005.\n"
          "Answer:")

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=40, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
answer = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("answer:", repr(answer.strip()))
assert answer.strip(), "merged model produced nothing - do not publish"

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUT, safe_serialization=True)
tokenizer.save_pretrained(OUT)
(OUT / "README.md").write_text(model_card(REPO_ID, BASE_MODEL), encoding="utf-8")

size = sum(f.stat().st_size for f in OUT.rglob("*") if f.is_file())
print(f"merged model: {size / 1e9:.1f} GB")
print((OUT / "README.md").read_text(encoding="utf-8")[:600])

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(REPO_ID, exist_ok=True, repo_type="model")
api.upload_folder(folder_path=str(OUT), repo_id=REPO_ID, repo_type="model")
print(f"published: https://huggingface.co/{REPO_ID}")

In [ ]:
# Also publish the adapter itself - 195 MB against 6.4 GB, and it is the artifact that was
# actually trained. Useful to anyone who already has the base model locally.
ADAPTER_REPO = REPO_ID + "-lora"
api.create_repo(ADAPTER_REPO, exist_ok=True, repo_type="model")
api.upload_folder(folder_path=str(ADAPTER_DIR), repo_id=ADAPTER_REPO, repo_type="model")
print(f"adapter published: https://huggingface.co/{ADAPTER_REPO}")